# Analysis 20 — who talks together?

Which friends end up in the same conversation most often? (log: `analysis-log.md`, Analysis 20).

Choices so far (student): a conversation ends after 12 hours without any message in the group chat;
a person takes part with at least 1 message, and again with at least 2; the graph is built both ways and compared. A pair counts only if it shares clearly more
conversations than shuffled versions that keep each person's number of conversations and each
conversation's size.

In [ ]:
import pandas as pd

from wa_analyzer.data import load_own_chat

GAP_HOURS = 12  # silence that ends a conversation (student; was 6)
MIN_MESSAGES = 2  # messages needed to take part in a conversation (student)

own = load_own_chat().sort_values("timestamp").reset_index(drop=True)

## Gap to the previous message

How long since the message before it, whoever wrote it.

In [ ]:
# Time since the previous message, whoever wrote it (NaT for the first message)
own["time_diff"] = own["timestamp"].diff()

## Conversations

A new conversation starts where the gap is longer than `GAP_HOURS`, and at the first message.

In [ ]:
# Label conversations: a new one starts after a gap longer than GAP_HOURS.
# The first message has NaT, which is not > GAP_HOURS, so it starts conversation 0.
own["conversation_id"] = (own["time_diff"] > pd.Timedelta(hours=GAP_HOURS)).cumsum()

# Number of messages per conversation
conversation_counts = own["conversation_id"].value_counts()

## Who takes part

Messages per author per conversation; keep authors with at least `MIN_MESSAGES`.

In [ ]:
# Number of messages per author per conversation
author_counts = (
    own.groupby(["conversation_id", "author"]).size().reset_index(name="message_count")
)

# Keep authors with at least MIN_MESSAGES messages in a conversation: one row per participant
participants = author_counts[author_counts["message_count"] >= MIN_MESSAGES]

## First numbers

- How many conversations?
- How many people per conversation (the distribution, not only the mean)?
- How many conversations have fewer than 2 people left (no links)?
- How many people does the 2-message rule drop, compared with 1 message?

In [ ]:
conversation_ids = own["conversation_id"].unique()
n_conversations = len(conversation_ids)

# People per conversation, with and without the 2-message rule. Conversations where nobody
# reached MIN_MESSAGES are missing from `participants`, so reindex them back in as 0.
people_any = (
    author_counts.groupby("conversation_id")
    .size()
    .reindex(conversation_ids, fill_value=0)
)
people_min = (
    participants.groupby("conversation_id")
    .size()
    .reindex(conversation_ids, fill_value=0)
)

print(f"Conversations: {n_conversations:,}")
print(
    f"Conversations with fewer than 2 participants (no links): {(people_min < 2).sum():,}"
)
print(
    f"Participations dropped by the {MIN_MESSAGES}-message rule: "
    f"{(people_any - people_min).sum():,} of {people_any.sum():,}"
)

# The spread, not only the mean: how many conversations have 0, 1, 2, ... participants
pd.DataFrame(
    {
        "at least 1 message": people_any.value_counts(),
        f"at least {MIN_MESSAGES} messages": people_min.value_counts(),
    }
).fillna(0).astype(int).sort_index().rename_axis("participants")

## Shared conversations per pair

For each pair of authors: in how many conversations did both take part? Built twice, with at least
1 message and with at least `MIN_MESSAGES`. These are raw counts: active people share more
conversations just by being active, so this is not yet the shuffle baseline from Stage 1.

In [ ]:
from itertools import combinations


def shared_conversations(people: pd.DataFrame) -> pd.Series:
    """Number of conversations each pair of authors took part in together."""
    pairs = [
        pair
        for _, authors in people.groupby("conversation_id")["author"]
        for pair in combinations(sorted(authors), 2)
    ]
    index = pd.MultiIndex.from_tuples(pairs, names=["author_a", "author_b"])
    return pd.Series(1, index=index).groupby(level=[0, 1]).sum()


RULES = {"at least 1 message": 1, f"at least {MIN_MESSAGES} messages": MIN_MESSAGES}
shared = (
    pd.DataFrame(
        {
            rule: shared_conversations(
                author_counts[author_counts["message_count"] >= n]
            )
            for rule, n in RULES.items()
        }
    )
    .fillna(0)
    .astype(int)
)

# The two rules give different totals, so compare each pair's share of all pair links, and its rank
share = shared / shared.sum()
rank = shared.rank(ascending=False, method="min").astype(int)

print(
    f"Pairs with at least one shared conversation: {(shared > 0).sum().to_dict()} of {len(shared)}"
)
print(
    f"Spearman rank correlation between the two rules: {shared.corr(method='spearman').iloc[0, 1]:.2f}"
)
share.round(3).join(rank, rsuffix=" (rank)").sort_values(
    list(RULES)[1], ascending=False
)

## The two graphs side by side

Draft. Same circular layout and author order in both panels, so only the edges differ. Edge width is a
pair's share of all pair links in that panel, so the two panels are on the same scale.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

authors = sorted(own["author"].unique())
positions = nx.circular_layout(authors)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, rule in zip(axes, RULES):
    graph = nx.Graph()
    graph.add_nodes_from(authors)
    for (a, b), value in share[rule].items():
        if value > 0:
            graph.add_edge(a, b, weight=value)
    widths = [graph[a][b]["weight"] * 150 for a, b in graph.edges]
    nx.draw_networkx_edges(graph, positions, width=widths, alpha=0.5, ax=ax)
    nx.draw_networkx_nodes(graph, positions, node_size=300, ax=ax)
    nx.draw_networkx_labels(graph, positions, font_size=9, ax=ax)
    ax.set_title(rule)
    ax.axis("off")
plt.tight_layout()

## Shared conversations compared with chance

With the 2-message rule (student). Expected shared conversations for a pair if people joined
conversations independently: conversations of A × conversations of B ÷ conversations. Ratio = observed ÷
expected: above 1 means together more often than their activity explains.

Choice to check: "conversations" here counts only conversations where at least one person took part under
the rule. This quick version does not keep conversation sizes; the shuffle from Stage 1 does.

In [ ]:
conversations_per_author = participants.groupby("author")["conversation_id"].nunique()
n_used = participants["conversation_id"].nunique()

observed = shared[list(RULES)[1]]
a = observed.index.get_level_values("author_a")
b = observed.index.get_level_values("author_b")
expected = (
    conversations_per_author[a].to_numpy()
    * conversations_per_author[b].to_numpy()
    / n_used
)
ratio = (observed / expected).rename("ratio")

print(f"Conversations used: {n_used:,}")
print(conversations_per_author.sort_values(ascending=False).to_string())
ratio.sort_values(ascending=False).round(2).to_frame()

## Heatmap

Draft. Authors ordered by hierarchical clustering on the ratio (distance = 1 ÷ ratio, average linkage), so
people who are together more often than chance sit next to each other and groups show as blocks.

In [ ]:
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

matrix = ratio.unstack()
matrix = matrix.combine_first(matrix.T).reindex(index=authors, columns=authors)

distance = (1 / matrix).fillna(0).to_numpy()
order = [
    authors[i]
    for i in leaves_list(linkage(squareform(distance, checks=False), method="average"))
]
ordered = matrix.loc[order, order]

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    ordered,
    mask=np.eye(len(order), dtype=bool),
    cmap="RdBu_r",
    center=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "shared conversations ÷ expected by chance"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Who ends up in the same conversations, compared with chance")
plt.tight_layout()

## Shuffle baseline

The baseline from Stage 1. Shuffle who is in which conversation while keeping each person's number of
conversations and each conversation's size (the curveball algorithm: two people at a time swap the
conversations only one of them was in). Expected = mean shared conversations over the shuffles.
Ratio = observed ÷ expected; share above = how often a shuffle gave at least the observed count.

In [ ]:
SEED = 42
N_SHUFFLES = 1_000
TRADES_PER_SHUFFLE = 200


def curveball_trade(memberships: list[set], rng: np.random.Generator) -> None:
    """Swap the conversations two random people do not share; both keep their number of conversations."""
    i, j = rng.choice(len(memberships), size=2, replace=False)
    both = memberships[i] & memberships[j]
    only_i, only_j = memberships[i] - both, memberships[j] - both
    pool = list(only_i | only_j)
    rng.shuffle(pool)
    memberships[i] = both | set(pool[: len(only_i)])
    memberships[j] = both | set(pool[len(only_i) :])


def shared_counts(memberships: list[set]) -> np.ndarray:
    return np.array(
        [
            len(memberships[i] & memberships[j])
            for i, j in combinations(range(len(memberships)), 2)
        ]
    )


rng = np.random.default_rng(SEED)
memberships = [
    set(participants.loc[participants["author"] == author, "conversation_id"])
    for author in authors
]
pair_index = pd.MultiIndex.from_tuples(
    list(combinations(authors, 2)), names=["author_a", "author_b"]
)
observed_counts = shared_counts(memberships)

shuffled = []
for _ in range(N_SHUFFLES):
    for _ in range(TRADES_PER_SHUFFLE):
        curveball_trade(memberships, rng)
    shuffled.append(shared_counts(memberships))
shuffled = np.array(shuffled)

baseline = pd.DataFrame(
    {
        "observed": observed_counts,
        "expected": shuffled.mean(axis=0),
        "share above": (shuffled >= observed_counts).mean(axis=0),
    },
    index=pair_index,
)
baseline["ratio"] = baseline["observed"] / baseline["expected"]
baseline.sort_values("ratio", ascending=False).round(3)

## Heatmap against the shuffle

Draft. Same ordering method as above, now on the shuffle ratio.

In [ ]:
shuffle_matrix = baseline["ratio"].unstack()
shuffle_matrix = shuffle_matrix.combine_first(shuffle_matrix.T).reindex(
    index=authors, columns=authors
)

distance = (1 / shuffle_matrix).fillna(0).to_numpy()
order = [
    authors[i]
    for i in leaves_list(linkage(squareform(distance, checks=False), method="average"))
]
ordered = shuffle_matrix.loc[order, order]
spread = np.nanmax(np.abs(ordered.to_numpy() - 1))

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    ordered,
    mask=np.eye(len(order), dtype=bool),
    cmap="RdBu_r",
    center=1,
    vmin=1 - spread,
    vmax=1 + spread,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "shared conversations ÷ expected from shuffles"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title(
    "Who ends up in the same conversations, compared with shuffled conversations"
)
plt.tight_layout()

## Weekend vs weekday

Test of the student's explanation: the core of four comes from meeting for weekend drinks. If so, the core
should stand out more in weekend conversations than in weekday ones. Weekend (student): a conversation that
starts from Friday 12:00 up to and including Sunday (timestamps are Amsterdam local time). The same shuffle
is run separately on each half.

In [ ]:
starts = own.groupby("conversation_id")["timestamp"].min()
is_weekend = (starts.dt.dayofweek >= 5) | (
    (starts.dt.dayofweek == 4) & (starts.dt.hour >= 12)
)
weekend_ids = set(starts.index[is_weekend])


def shuffle_baseline(people: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    """Observed and shuffled shared conversations per pair, for one set of participants."""
    rng = np.random.default_rng(seed)
    memberships = [
        set(people.loc[people["author"] == author, "conversation_id"])
        for author in authors
    ]
    observed = shared_counts(memberships)
    runs = []
    for _ in range(N_SHUFFLES):
        for _ in range(TRADES_PER_SHUFFLE):
            curveball_trade(memberships, rng)
        runs.append(shared_counts(memberships))
    runs = np.array(runs)
    result = pd.DataFrame(
        {
            "observed": observed,
            "expected": runs.mean(axis=0),
            "share above": (runs >= observed).mean(axis=0),
        },
        index=pair_index,
    )
    result["ratio"] = result["observed"] / result["expected"]
    return result


in_weekend = participants["conversation_id"].isin(weekend_ids)
halves = {"weekend": participants[in_weekend], "weekday": participants[~in_weekend]}
for name, people in halves.items():
    print(
        f"{name}: {people['conversation_id'].nunique():,} conversations with a participant"
    )

split = pd.concat(
    {name: shuffle_baseline(people) for name, people in halves.items()}, axis=1
)
core = ["effervescent-penguin", "fluffy-beaver", "rib-tickling-curlew", "striking-rail"]
in_core = [a in core and b in core for a, b in split.index]
split.insert(0, "core pair", in_core)
split.sort_values(("weekend", "ratio"), ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
both = pd.concat([split[(name, "ratio")] for name in halves])
spread = np.nanmax(np.abs(both.to_numpy() - 1))
for ax, name in zip(axes, halves):
    m = split[(name, "ratio")].unstack()
    m = m.combine_first(m.T).reindex(index=authors, columns=authors).loc[order, order]
    sns.heatmap(
        m,
        mask=np.eye(len(order), dtype=bool),
        cmap="RdBu_r",
        center=1,
        vmin=1 - spread,
        vmax=1 + spread,
        annot=True,
        fmt=".2f",
        square=True,
        linewidths=0.5,
        cbar=ax is axes[-1],
        cbar_kws={"label": "shared conversations ÷ expected from shuffles"},
        ax=ax,
    )
    ax.set_title(f"{name} conversations")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()

## Planning label check

The weekend split measures *when*, the explanation is about *what* (student). Planning messages start the
conversations about who shows up, so the unit is the message (student). The pattern is Analysis 2's
`is_planning`: *wie/iemand* AND an activity word.

Two samples of 50 to label by hand (`y`/`n`), as in the football label check:
- **matched**: flagged by the pattern, to measure how many really are planning;
- **near miss**: not flagged, but containing one of the student's words (*wie, iemand, vnv, biertje, drinken*)
  or an activity word from the pattern, to measure what the pattern misses.

Written to `data/processed/` (gitignored: real messages). Never overwritten, so labels survive a rerun.

In [ ]:
import re

from wa_analyzer.data import PROCESSED

WHO = r"(?i)\b(?:wie|iemand)\b"
ACTIVITY = (
    r"(?i)\b(?:vnv|afspreken|vanavond|weekend|wat\s+doen|wat\s+drinken|iets\s+doen)\b"
)
NEAR = r"(?i)\b(?:wie|iemand|vnv|biertjes?|drinken|afspreken|vanavond|weekend|wat\s+doen|iets\s+doen)\b"
GLUED_SYSTEM_LINE = re.compile(
    r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - "
)  # parser glitch; carries real names
NOT_A_TEXT = re.compile(
    r"<Media weggelaten>|<Video note omitted>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand"
)
LABEL_FILE = PROCESSED / "planning-label-check.csv"
SAMPLE_SIZE = 50

text = own["message"].fillna("").astype(str)
is_text = ~text.str.contains(GLUED_SYSTEM_LINE) & ~text.str.contains(NOT_A_TEXT)
own["is_planning"] = text.str.contains(WHO) & text.str.contains(ACTIVITY)
near_miss = is_text & ~own["is_planning"] & text.str.contains(NEAR)

print(
    f"Matched: {(is_text & own['is_planning']).sum():,} messages; near-miss pool: {near_miss.sum():,}"
)

if LABEL_FILE.exists():
    print(f"{LABEL_FILE.name} exists, not overwritten")
else:
    pools = {"matched": own[is_text & own["is_planning"]], "near miss": own[near_miss]}
    samples = pd.concat(
        [
            pool.sample(min(SAMPLE_SIZE, len(pool)), random_state=SEED).assign(
                sample=name
            )
            for name, pool in pools.items()
        ]
    )
    samples.assign(is_planning_label="")[
        ["sample", "timestamp", "message", "is_planning_label"]
    ].to_csv(LABEL_FILE, index=False)
    print(f"Wrote {len(samples)} rows to {LABEL_FILE}")

## Widened planning pattern

Label check result (student labels, 100 messages): the pattern is precise (47 of 50 matched are planning)
but finds about 35% of planning messages at most (11 of 50 near misses are planning). Most misses have
*wie/iemand* but no activity word from the list.

Added (student): next to *wie/iemand*, day and time words (A), *komen/mee* (B), *samen* (C) and drinks (D);
on their own, *waar doen/zullen/gaan/willen we* questions (F). Not added: *kijken* (E).

A new sample of 50 from only the added matches measures whether the additions are planning too.
Written to its own file, so the first labels stay untouched.

In [ ]:
EXTRA = (
    r"(?i)\b(?:(?:maan|dins|woens|donder|vrij|zater|zon)dag(?:avond|middag|ochtend)?|morgen(?:avond|middag)?|straks"  # A
    r"|komen|komt|mee"  # B
    r"|samen"  # C
    r"|bier|biertjes?|pils(?:je|jes)?|drankjes?|potten)\b"  # D
)
WHERE = r"(?i)\bwaar\s+(?:doen|zullen|gaan|willen)\s+we\b"  # F: no wie/iemand needed
WIDE_LABEL_FILE = PROCESSED / "planning-label-check-wide.csv"

own["is_planning_wide"] = (
    own["is_planning"]
    | (text.str.contains(WHO) & text.str.contains(EXTRA))
    | text.str.contains(WHERE)
)
added = is_text & own["is_planning_wide"] & ~own["is_planning"]

print(
    f"Widened: {(is_text & own['is_planning_wide']).sum():,} messages; added: {added.sum():,}"
)

if WIDE_LABEL_FILE.exists():
    print(f"{WIDE_LABEL_FILE.name} exists, not overwritten")
else:
    wide_sample = (
        own[added]
        .sample(min(SAMPLE_SIZE, added.sum()), random_state=SEED)
        .assign(sample="added")
    )
    wide_sample.assign(is_planning_label="")[
        ["sample", "timestamp", "message", "is_planning_label"]
    ].to_csv(WIDE_LABEL_FILE, index=False)
    print(f"Wrote {len(wide_sample)} rows to {WIDE_LABEL_FILE}")

## Planning vs other conversations

Test of the student's explanation with topic instead of time: if the core of four comes from meeting up,
it should stand out more in conversations about planning. A conversation counts as planning when its first
planning message (widened pattern) is within its first `PLANNING_WITHIN` messages (student): a planning
message later on makes it hard to say the whole conversation is about planning. The same shuffle is run
separately on each half.

In [ ]:
PLANNING_WITHIN = (
    10  # first planning message within this many messages of the conversation (student)
)

position = own.groupby("conversation_id").cumcount() + 1
first_planning = (
    position[is_text & own["is_planning_wide"]].groupby(own["conversation_id"]).min()
)
planning_ids = set(first_planning.index[first_planning <= PLANNING_WITHIN])

in_planning = participants["conversation_id"].isin(planning_ids)
topics = {"planning": participants[in_planning], "other": participants[~in_planning]}
for name, people in topics.items():
    print(
        f"{name}: {people['conversation_id'].nunique():,} conversations with a participant"
    )

topic_split = pd.concat(
    {name: shuffle_baseline(people) for name, people in topics.items()}, axis=1
)
topic_split.insert(0, "core pair", in_core)
topic_split.sort_values(("planning", "ratio"), ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
both = pd.concat([topic_split[(name, "ratio")] for name in topics])
spread = np.nanmax(np.abs(both.to_numpy() - 1))
for ax, name in zip(axes, topics):
    m = topic_split[(name, "ratio")].unstack()
    m = m.combine_first(m.T).reindex(index=authors, columns=authors).loc[order, order]
    sns.heatmap(
        m,
        mask=np.eye(len(order), dtype=bool),
        cmap="RdBu_r",
        center=1,
        vmin=1 - spread,
        vmax=1 + spread,
        annot=True,
        fmt=".2f",
        square=True,
        linewidths=0.5,
        cbar=ax is axes[-1],
        cbar_kws={"label": "shared conversations ÷ expected from shuffles"},
        ax=ax,
    )
    ax.set_title(f"{name} conversations")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()

## Distance and planning

Student's question: one might assume distance matters for planning, but some people may travel far to see
their friends. Prediction (student, before running): people take part in planning conversations similarly
when they live further apart; the only outlier would be humorous-stingray, who lives abroad. Taking part
includes saying "ik kan niet": this data does not show who says yes.

Locations (student): a town per person with the year they moved there, in gitignored
`data/processed/author-location.csv` (friends' personal data, town level only; a year means 1 January).
Distance = straight-line km between the two towns on the date a conversation starts; per pair the median
over the planning conversations. The 36 pairs share people, so the test shuffles the 9 people's locations
(Mantel test) instead of treating pairs as independent.

In [ ]:
from scipy.stats import spearmanr

LOCATION_FILE = PROCESSED / "author-location.csv"
N_LOCATION_SHUFFLES = 10_000
TOWNS = {  # town centre: latitude, longitude
    "Losser": (52.261, 7.004),
    "Oldenzaal": (52.313, 6.929),
    "Deventer": (52.255, 6.163),
    "Groningen": (53.219, 6.567),
    "Amsterdam": (52.370, 4.895),
    "Leiden": (52.160, 4.497),
    "Vienna": (48.208, 16.373),
}

locations = pd.read_csv(LOCATION_FILE, parse_dates=["since"]).sort_values("since")


def town_on(author: str, date: pd.Timestamp) -> str:
    """The town an author lived in on a date: the last move on or before it."""
    moves = locations[(locations["author"] == author) & (locations["since"] <= date)]
    return moves["town"].iloc[-1]


def km_between(town_a: str, town_b: str) -> float:
    """Straight-line distance (haversine) between two town centres."""
    (lat_a, lon_a), (lat_b, lon_b) = (
        np.radians(TOWNS[town_a]),
        np.radians(TOWNS[town_b]),
    )
    h = (
        np.sin((lat_b - lat_a) / 2) ** 2
        + np.cos(lat_a) * np.cos(lat_b) * np.sin((lon_b - lon_a) / 2) ** 2
    )
    return 2 * 6371 * np.arcsin(np.sqrt(h))


planning_starts = starts[starts.index.isin(planning_ids)]
pair_km = pd.Series(
    [
        np.median(
            [km_between(town_on(a, date), town_on(b, date)) for date in planning_starts]
        )
        for a, b in pair_index
    ],
    index=pair_index,
    name="km",
)

distance_table = pd.DataFrame(
    {
        "km": pair_km,
        "planning ratio": topic_split[("planning", "ratio")],
        "other ratio": topic_split[("other", "ratio")],
    }
)


def location_shuffle_test(
    km: pd.Series, ratio: pd.Series, seed: int = SEED
) -> tuple[float, float]:
    """Spearman correlation of km with ratio, and how often shuffled locations give one at least as far from 0."""
    rng = np.random.default_rng(seed)
    km_matrix = km.unstack()
    km_matrix = (
        km_matrix.combine_first(km_matrix.T)
        .reindex(index=authors, columns=authors)
        .fillna(0)
        .to_numpy()
    )
    pairs = [(authors.index(a), authors.index(b)) for a, b in pair_index]
    observed = spearmanr(km, ratio).statistic
    shuffled = []
    for _ in range(N_LOCATION_SHUFFLES):
        p = rng.permutation(len(authors))
        shuffled.append(
            spearmanr([km_matrix[p[i], p[j]] for i, j in pairs], ratio).statistic
        )
    return observed, (np.abs(shuffled) >= abs(observed)).mean()


for half in ["planning ratio", "other ratio"]:
    rho, share = location_shuffle_test(pair_km, distance_table[half])
    print(
        f"{half}: Spearman {rho:+.2f}, share of location shuffles at least as strong {share:.3f}"
    )

distance_table.sort_values("km").round(2)

## Planning network on a distance map (2023–2026)

Draft. One sentence (student's result): planning together does not fade with distance.

- **Period (student):** planning conversations from 2023 on, when most moves had happened; the shuffle is
  rerun for this period only.
- **Position:** each person at the town they lived in for most of these conversations. Angle = compass
  direction from Losser, radius = distance from Losser on a log scale (student: Losser at the centre).
  People in the same town are spread a little so they do not overlap.
- **Edges:** every pair, also the low ones (student). Colour and width = planning ratio against the shuffle,
  same red/blue scale as the heatmaps: red above chance, blue below.

In [ ]:
from matplotlib.colors import TwoSlopeNorm

PERIOD_START = pd.Timestamp("2023-01-01")  # student

period_ids = set(planning_starts.index[planning_starts >= PERIOD_START])
period_people = participants[participants["conversation_id"].isin(period_ids)]
period = shuffle_baseline(period_people)
print(
    f"Planning conversations from {PERIOD_START:%Y}: {period_people['conversation_id'].nunique():,}"
)

# Each person's town: the one they lived in for most of these conversations
period_dates = planning_starts[planning_starts.index.isin(period_ids)]
home = {
    author: pd.Series([town_on(author, date) for date in period_dates]).mode()[0]
    for author in authors
}


def map_position(town: str) -> np.ndarray:
    """Compass direction from Losser as angle, log10(1 + km from Losser) as radius."""
    lat0, lon0 = TOWNS["Losser"]
    lat, lon = TOWNS[town]
    angle = np.arctan2(lat - lat0, (lon - lon0) * np.cos(np.radians(lat0)))
    radius = np.log10(1 + km_between("Losser", town))
    return radius * np.array([np.cos(angle), np.sin(angle)])


SPREAD = 0.4  # spacing between people in the same town
positions_map, label_direction = {}, {}
for town, group in pd.Series(home).groupby(lambda author: home[author]):
    centre = map_position(town)
    # Spread along the ring (sideways), so same-town people do not point at the next town in
    first_turn = 0 if town == "Losser" else np.arctan2(centre[1], centre[0]) + np.pi / 2
    for k, author in enumerate(sorted(group.index)):
        turn = 2 * np.pi * k / len(group) + first_turn
        offset = (
            SPREAD * np.array([np.cos(turn), np.sin(turn)])
            if len(group) > 1
            else np.zeros(2)
        )
        positions_map[author] = centre + offset
        outward = (
            offset if len(group) > 1 else centre
        )  # labels point away from the town, or from Losser
        label_direction[author] = outward / np.linalg.norm(outward)

period_km = pd.Series(
    [km_between(home[a], home[b]) for a, b in pair_index], index=pair_index
)
rho, share = location_shuffle_test(period_km, period["ratio"])
print(
    f"2023–2026 planning ratio vs km: Spearman {rho:+.2f}, share of location shuffles at least as strong {share:.3f}"
)

edges = period["ratio"].reindex(
    period["ratio"].sub(1).abs().sort_values().index
)  # strongest drawn last
spread = edges.sub(1).abs().max()
norm = TwoSlopeNorm(vmin=1 - spread, vcenter=1, vmax=1 + spread)
cmap = plt.get_cmap("RdBu_r")

fig, ax = plt.subplots(figsize=(12, 10.5))
for km in [10, 50, 100, 500]:
    r = np.log10(1 + km)
    ax.add_patch(
        plt.Circle((0, 0), r, fill=False, color="0.8", linestyle="--", linewidth=0.8)
    )
    ax.text(0, r, f"{km} km", ha="center", va="bottom", fontsize=8, color="0.5")
for (a, b), value in edges.items():
    (xa, ya), (xb, yb) = positions_map[a], positions_map[b]
    ax.plot(
        [xa, xb],
        [ya, yb],
        color=cmap(norm(value)),
        linewidth=1 + 30 * abs(value - 1),
        solid_capstyle="round",
    )
for author, (x, y) in positions_map.items():
    ax.scatter(x, y, s=120, color="0.25", zorder=3)
    dx, dy = label_direction[author]
    ax.annotate(
        f"{author}\n({home[author]})",
        (x, y),
        xytext=(14 * dx, 14 * dy),
        textcoords="offset points",
        ha="left" if dx > 0.3 else "right" if dx < -0.3 else "center",
        va="bottom" if dy > 0.3 else "top" if dy < -0.3 else "center",
        fontsize=8,
        bbox={
            "boxstyle": "round,pad=0.2",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.8,
        },
        zorder=4,
    )
ax.text(0, 0, "Losser", ha="center", va="center", fontsize=9, color="0.5")
fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    shrink=0.6,
    label="planning conversations together ÷ expected from shuffles",
)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(
    "Planning together does not fade with distance (planning conversations, 2023–2026)"
)
plt.tight_layout()

period.sort_values("ratio", ascending=False).round(3)

## When A invites, who joins? (2023–2026)

Student: the ratio against shuffles needs too much thinking for a first-time reader. Simpler measure: for the
planning conversations person A started (A wrote the first planning message), the share in which person B took
part (at least `MIN_MESSAGES` messages). One arrow per direction. The shuffle's correction for activity is
dropped; instead each person's overall join rate is shown next to their name, so a reader can see who joins a
lot anyway.

Two layouts (student): Losser at the centre as before, and an alternative where line length is the
distance between the two people (log km; classical multidimensional scaling, so approximately).

In [ ]:
from matplotlib.patches import FancyArrowPatch

period_planning = own[
    own["conversation_id"].isin(period_ids) & is_text & own["is_planning_wide"]
]
inviter = (
    period_planning.sort_values("timestamp")
    .groupby("conversation_id")["author"]
    .first()
)
joined = period_people.groupby("conversation_id")["author"].apply(set)

invites = inviter.value_counts().reindex(authors, fill_value=0)
join_rate = pd.Series(
    {b: np.mean([b in joined.get(c, set()) for c in inviter.index]) for b in authors}
)
share_joined = pd.Series(
    {
        (a, b): np.mean(
            [b in joined.get(c, set()) for c in inviter.index[inviter == a]]
        )
        for a in authors
        for b in authors
        if a != b and invites[a] > 0
    }
).rename_axis(["inviter", "joiner"])

print(
    pd.DataFrame(
        {"invites started": invites, "joins overall": join_rate.round(2)}
    ).to_string()
)

# Alternative layout: line length = distance between the two people (log km)
log_km = np.array(
    [[np.log10(1 + km_between(home[a], home[b])) for b in authors] for a in authors]
)
centring = np.eye(len(authors)) - 1 / len(authors)
values, vectors = np.linalg.eigh(-0.5 * centring @ log_km**2 @ centring)
mds = vectors[:, -2:][:, ::-1] * np.sqrt(values[-2:][::-1])
if (
    mds[authors.index(home_author := "humorous-stingray"), 0] < 0
):  # Vienna to the east, as on a map
    mds[:, 0] *= -1
positions_pair = {}
for town, group in pd.Series(home).groupby(lambda author: home[author]):
    centre = mds[authors.index(group.index[0])]
    for k, author in enumerate(sorted(group.index)):
        turn = 2 * np.pi * k / len(group)
        positions_pair[author] = centre + (
            SPREAD * np.array([np.cos(turn), np.sin(turn)]) if len(group) > 1 else 0
        )

SUBTITLE = (
    "Arrow A → B: share of the planning conversations A started in which B took part"
)
cmap_share = plt.get_cmap("Reds")


def draw_invite_network(ax: plt.Axes, positions: dict, rings: bool) -> None:
    if rings:
        for km in [10, 50, 100, 500]:
            r = np.log10(1 + km)
            ax.add_patch(
                plt.Circle(
                    (0, 0), r, fill=False, color="0.85", linestyle="--", linewidth=0.8
                )
            )
            ax.text(
                0,
                r,
                f"{km} km from Losser",
                ha="center",
                va="bottom",
                fontsize=7,
                color="0.5",
            )
    for (a, b), value in share_joined.sort_values().items():
        ax.add_patch(
            FancyArrowPatch(
                positions[a],
                positions[b],
                connectionstyle="arc3,rad=0.15",
                arrowstyle="-|>",
                mutation_scale=10,
                color=cmap_share(value),
                linewidth=0.5 + 5 * value,
                shrinkA=8,
                shrinkB=8,
            )
        )
    centre = np.mean(list(positions.values()), axis=0)
    for author, xy in positions.items():
        ax.scatter(*xy, s=110, color="0.25", zorder=3)
        direction = (xy - centre) / (np.linalg.norm(xy - centre) or 1)
        ax.annotate(
            f"{author} ({home[author]})\njoins {join_rate[author]:.0%} · started {invites[author]}",
            xy,
            xytext=(16 * direction[0], 16 * direction[1]),
            textcoords="offset points",
            ha="left"
            if direction[0] > 0.3
            else "right"
            if direction[0] < -0.3
            else "center",
            va="bottom"
            if direction[1] > 0.3
            else "top"
            if direction[1] < -0.3
            else "center",
            fontsize=7.5,
            bbox={
                "boxstyle": "round,pad=0.2",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.85,
            },
            zorder=4,
        )
    ax.set_aspect("equal")
    ax.axis("off")


fig, axes = plt.subplots(1, 2, figsize=(20, 10))
draw_invite_network(axes[0], positions_map, rings=True)
axes[0].set_title(
    "Losser at the centre: distance from the centre = distance from Losser"
)
draw_invite_network(axes[1], positions_pair, rings=False)
axes[1].set_title("Alternative: line length = distance between the two people")
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap_share),
    ax=axes,
    shrink=0.5,
    label="share of A's invites that B joined",
)
fig.suptitle(
    f"Planning together does not fade with distance (2023–2026)\n{SUBTITLE}",
    fontsize=13,
)

share_joined.unstack().round(2)

## Draft for 3 Nov: living close is not the same as being close

Story (student): the group met in Losser; half moved away, yet distance does not decide who plans together. The
edges carry the story (network theme: who connects, who drifts): every pair in grey (all data: every pair
shares planning conversations, at least 18), coloured where the pair clearly differs from the shuffle (share
above ≤ `CLEAR` or ≥ 1 − `CLEAR`), thicker the further from chance. Dots are equal so they do not compete with
the edges; each label gives the take-part rate. No names and no towns (student): labels give km from the hometown (Losser, where everyone grew up), shown as
"Hometown".
Taking part means the conversation, not saying yes or coming.

In [ ]:
from matplotlib.lines import Line2D

CLEAR = 0.05
clear_pairs = period[
    (period["share above"] <= CLEAR) | (period["share above"] >= 1 - CLEAR)
]
MORE, LESS = cmap(0.85), cmap(0.15)
# Red bends one way, blue the other, so lines to the same dot do not run together;
# the Vienna–Oldenzaal line goes around the right instead of through Losser (student)
BEND = {("humorous-stingray", "vibrant-barracuda"): 0.5}

fig, ax = plt.subplots(figsize=(15, 11))
for km in [10, 50, 100, 500]:
    r = np.log10(1 + km)
    ax.add_patch(
        plt.Circle((0, 0), r, fill=False, color="0.85", linestyle="--", linewidth=0.8)
    )
    ax.text(
        0,
        r,
        f"{km} km from hometown",
        ha="center",
        va="bottom",
        fontsize=8,
        color="0.5",
    )
for a, b in pair_index:  # every pair, in the background
    ax.add_patch(
        FancyArrowPatch(
            positions_map[a],
            positions_map[b],
            connectionstyle="arc3,rad=0.25",
            arrowstyle="-",
            color="0.85",
            linewidth=1,
        )
    )
for (a, b), row in clear_pairs.iterrows():
    bend = BEND.get((a, b), 0.25 if row["ratio"] > 1 else -0.25)
    ax.add_patch(
        FancyArrowPatch(
            positions_map[a],
            positions_map[b],
            connectionstyle=f"arc3,rad={bend}",
            arrowstyle="-",
            color=MORE if row["ratio"] > 1 else LESS,
            linewidth=2 + 25 * abs(row["ratio"] - 1),
            zorder=2,
        )
    )
km_from_hometown = {
    author: km_between("Losser", home[author]) for author in authors
}  # everyone grew up in Losser
for author, (x, y) in positions_map.items():
    ax.scatter(x, y, s=150, color="0.3", edgecolor="white", linewidth=1.5, zorder=3)
    dx, dy = label_direction[author]
    ax.annotate(
        f"{join_rate[author]:.0%}"
        if home[author] == "Losser"
        else f"{km_from_hometown[author]:.0f} km\n{join_rate[author]:.0%}",
        (x, y),
        xytext=(16 * dx, 16 * dy),
        textcoords="offset points",
        ha="left" if dx > 0.3 else "right" if dx < -0.3 else "center",
        va="bottom" if dy > 0.3 else "top" if dy < -0.3 else "center",
        fontsize=8,
        bbox={
            "boxstyle": "round,pad=0.2",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.85,
        },
        zorder=4,
    )
ax.text(0, 0, "Hometown", ha="center", va="center", fontsize=10, color="0.35")

vienna, oldenzaal_pair = (
    ("humorous-stingray", "hypnotic-rabbit"),
    ("humorous-stingray", "vibrant-barracuda"),
)
ax.annotate(
    "800 km away, yet the group's strongest tie:\n"
    f"{period.loc[vienna, 'ratio'] - 1:.0%} more often than chance with an author in the hometown,\n"
    f"{1 - period.loc[oldenzaal_pair, 'ratio']:.0%} less often with the author {km_from_hometown['vibrant-barracuda']:.0f} km from it",
    positions_map["humorous-stingray"],
    xytext=(-30, -50),
    textcoords="offset points",
    ha="left",
    va="top",
    fontsize=10,
    arrowprops={"arrowstyle": "-", "color": "0.5", "linewidth": 0.8},
)
less_lines = clear_pairs[clear_pairs["ratio"] < 1].index
drifter = pd.Series([name for pair in less_lines for name in pair]).value_counts()
ax.annotate(
    f'Lives in the hometown, yet on {drifter.iloc[0]} of the {len(less_lines)} "less often" lines',
    positions_map[drifter.index[0]],
    xytext=(-60, -150),
    textcoords="offset points",
    ha="right",
    va="top",
    fontsize=10,
    arrowprops={"arrowstyle": "-", "color": "0.5", "linewidth": 0.8},
)

ax.legend(
    handles=[
        Line2D([], [], color=MORE, linewidth=5, label="more often than chance"),
        Line2D([], [], color=LESS, linewidth=5, label="less often than chance"),
        Line2D([], [], color=MORE, linewidth=2, label="thicker = further from chance"),
        Line2D([], [], color="0.85", linewidth=1, label="about as often as chance"),
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            color="0.3",
            markersize=8,
            label="author: km from hometown, % takes part",
        ),
    ],
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize=9,
)
fig.text(
    0.02,
    0.01,
    "Chance: the same people, equally active, shuffled 1,000 times over the conversations. A line is coloured when "
    "at most 5% of shuffles give a count this high (red) or this low (blue).\n"
    "With 36 pairs, about 2 coloured lines can be chance. 151 planning conversations. Taking part = at least 2 "
    'messages in the conversation, "ik kan niet" included: not saying yes, not coming.',
    fontsize=10,
    color="0.4",
    va="bottom",
)
ax.set_aspect("equal")
ax.axis("off")
fig.text(
    0.02, 0.975, "Living close is not the same as being close", fontsize=18, va="top"
)
fig.text(  # subtitle (student)
    0.02,
    0.94,
    "All nine grew up in the same hometown. Moving away didn't loosen the ties; staying didn't guarantee them "
    "(planning conversations 2023–2026)",
    fontsize=13,
    color="0.3",
    va="top",
)
plt.tight_layout(rect=(0, 0.06, 1, 0.92))
fig.savefig("network-final.png", dpi=200, bbox_inches="tight")

clear_pairs[["observed", "expected", "share above", "ratio"]].sort_values(
    "ratio", ascending=False
).round(3)